In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import warnings

warnings.filterwarnings('ignore')

!pip install openpyxl
path = '/content/Integrating AI-DataSet_Youth_Wellness.xlsx'
df = pd.read_excel(path, sheet_name="Sheet3")

In [ ]:
df.shape
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df['clean_text']
y = df['is_depression']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train.shape, X_test.shape

In [ ]:
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf_entropy = DecisionTreeClassifier(criterion="entropy", max_depth=10,
    min_samples_leaf=5,
    min_samples_split=5,
    class_weight= 'balanced',
    random_state=42
  )

clf_entropy.fit(X_train, y_train)
y_pred_entropy = clf_entropy.predict(X_test)

y_train_pred = clf_entropy.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

print("DT Accuracy(For Entropy):", accuracy_score(y_test, y_pred_entropy))
print("\nClassification Report:\n", classification_report(y_test, y_pred_entropy))

cm_entropy = confusion_matrix(y_test, y_pred_entropy)
plt.figure(figsize=(3.5,3))
sns.heatmap(cm_entropy, annot=True, fmt="d", cmap="Oranges", xticklabels=["Non-Depressed", "Depressed"], yticklabels=["Non-Depressed", "Depressed"])
plt.title("DT")
plt.show()

In [ ]:
clf_gini = DecisionTreeClassifier(criterion="gini", max_depth= 10, min_samples_leaf = 5, min_samples_split=5,
    class_weight= 'balanced', random_state=42)

clf_gini.fit(X_train, y_train)
y_pred_gini = clf_gini.predict(X_test)

y_train_pred = clf_gini.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("Training Accuracy:", accuracy_score(y_train, y_train_pred))

print("DT Accuracy(For gini):", accuracy_score(y_test, y_pred_gini))
print("\nClassification Report:\n", classification_report(y_test, y_pred_gini))

cm_gini = confusion_matrix(y_test, y_pred_gini)
plt.figure(figsize=(3.5,3))
sns.heatmap(cm_gini, annot=True, fmt="d", cmap="Greens", xticklabels=["Non-Depressed", "Depressed"], yticklabels=["Non-Depressed", "Depressed"])
plt.title("DT")
plt.show()

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.svm import LinearSVC

X_text = df['clean_text'].astype(str).values
y = df['is_depression'].values


X_train_text, X_test_text, y_train, y_test, train_idx, test_idx = train_test_split(
    X_text,
    y,
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_tr_text, X_val_text, y_tr, y_val = train_test_split(
    X_train_text, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [ ]:
from sklearn.metrics import accuracy_score

def objective(trial):

    max_features = trial.suggest_int("max_features", 2000, 7740, step=1000)
    ngram_range = trial.suggest_categorical("ngram_range", [(1, 1), (1, 2)])


    C = trial.suggest_float("C", 1e-3, 1e2, log=True)
    class_weight_choice = trial.suggest_categorical(
        "class_weight", ["none", "balanced"]
    )
    class_weight = None if class_weight_choice == "none" else "balanced"


    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range
    )

    X_tr_vec = vectorizer.fit_transform(X_tr_text)
    X_val_vec = vectorizer.transform(X_val_text)
    model = LinearSVC(
        C=C,
        class_weight=class_weight,
        random_state=42,
        max_iter=250
    )

    model.fit(X_tr_vec, y_tr)

    y_val_pred = model.predict(X_val_vec)
    val_acc = accuracy_score(y_val, y_val_pred)

    return val_acc

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best validation accuracy:", study.best_value)
print("Best params:", study.best_params)
print("Best validation accuracy:", study.best_value)


In [ ]:
best_params = study.best_params
best_C = best_params["C"]
best_class_weight = None if best_params["class_weight"] == "none" else "balanced"
best_max_features = best_params["max_features"]
best_ngram_range = best_params["ngram_range"]


final_vectorizer = TfidfVectorizer(
    max_features=best_max_features,
    ngram_range=best_ngram_range
)

X_train_vec = final_vectorizer.fit_transform(X_train_text)
X_test_vec = final_vectorizer.transform(X_test_text)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

final_model = LinearSVC(
    C=best_C,
    class_weight=best_class_weight,
    random_state=42,
    max_iter=10000
)
final_model.fit(X_train_vec, y_train)
y_test_pred = final_model.predict(X_test_vec)

test_acc = accuracy_score(y_test, y_test_pred)
cm = confusion_matrix(y_test, y_test_pred)

In [ ]:
from sklearn.metrics import classification_report

print("Training Accuracy:", accuracy_score(y_train, final_model.predict(X_train_vec)))
print(f"\nTest Accuracy: {test_acc:.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_test_pred))

w = final_model.coef_
w_norm = np.linalg.norm(w)
margin = 1.0 / w_norm
print(f"Approximate geometric margin: {margin:.6f}")


plt.figure(figsize=(3.5, 3))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    linewidths=1,
    linecolor='white',
    xticklabels=["Non-Depressed", "Depressed"],
    yticklabels=["Non-Depressed", "Depressed"]
)

plt.title(
    "SVM",
    fontsize=16, weight="bold"
)
plt.xlabel("Predicted", fontsize=14)
plt.ylabel("True", fontsize=14)
ax.xaxis.set_tick_params(labelsize=12)
ax.yaxis.set_tick_params(labelsize=12)
plt.show()

In [ ]:
y_pred = y_test_pred
mis_idx = np.where(y_pred != y_test)[0]

for i in mis_idx[:50]:
    print("TEXT:", X_test_text[i])
    print("True Label:", y_test[i], "| Predicted:", y_pred[i])
    print("-" * 80)


test_idx = np.array(test_idx)
mis_row_ids = test_idx[mis_idx]

#print(f"\nTotal misclassified samples in test set: {len(mis_row_ids)}")


df_cleaned = df.drop(index=mis_row_ids).reset_index(drop=True)

print(f"Original dataset size: {len(df)} rows")
print(f"Cleaned dataset size:  {len(df_cleaned)} rows")

X_text_clean = df_cleaned['clean_text'].astype(str).values
y_clean = df_cleaned['is_depression'].values


X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_text_clean,
    y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)


vectorizer_clean = TfidfVectorizer(
    max_features=best_max_features,
    ngram_range=best_ngram_range
)

X_train_vec_clean = vectorizer_clean.fit_transform(X_train_clean)
X_test_vec_clean = vectorizer_clean.transform(X_test_clean)


model_clean = LinearSVC(
    C=best_C,
    class_weight=best_class_weight,
    random_state=42,
    max_iter=10000
)

model_clean.fit(X_train_vec_clean, y_train_clean)
y_pred_clean = model_clean.predict(X_test_vec_clean)


cm_svm = confusion_matrix(y_test_clean, y_pred_clean)

plt.figure(figsize=(3.5, 3))
sns.heatmap(
    cm_svm,
    annot=True,
    fmt='d',
    cmap='Greens',
    linewidths=1,
    linecolor='white',
    xticklabels=["Non-Depressed", "Depressed"],
    yticklabels=["Non-Depressed", "Depressed"]
)

plt.title("SVM", fontsize=16, weight="bold")
plt.xlabel("Predicted", fontsize=14)
plt.ylabel("True", fontsize=14)
plt.show()

print("Accuracy on cleaned dataset:", accuracy_score(y_test_clean, y_pred_clean))

In [ ]:
!pip install -q textstat sentence-transformers

In [ ]:
import os, re
import nltk
import textstat
import torch
import joblib
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer

CONF = {
    "data_path": "/content/Integrating AI-DataSet_Youth_Wellness.xlsx",
    "save_dir": "processed_data",
    "sbert_model": "all-mpnet-base-v2",
    "tfidf_max_features": 5000,
}

In [ ]:
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

class DepressionFeatureStore:

    def __init__(self, config):
        self.config = config

    def load_and_clean(self):

        df = pd.read_excel(self.config["data_path"], sheet_name="Sheet3")
        df["clean_text"] = df["clean_text"].astype(str).fillna("")
        df["clean_text"] = df["clean_text"].apply(lambda x: " ".join(x.split()))
        self.df = df
        return self

    def extract_psycholinguistic(self):
        sid = SentimentIntensityAnalyzer()

        self.df["vader_compound"] = self.df["clean_text"].apply(lambda x: sid.polarity_scores(x)["compound"])
        self.df["vader_neg"] = self.df["clean_text"].apply(lambda x: sid.polarity_scores(x)["neg"])
        self.df["vader_pos"] = self.df["clean_text"].apply(lambda x: sid.polarity_scores(x)["pos"])

        def pronoun_ratio(t):
            w = t.lower().split()
            if not w: return 0
            p = len(re.findall(r"\b(i|me|my|mine|myself)\b", t.lower()))
            return p / len(w)

        self.df["pronoun_ratio"] = self.df["clean_text"].apply(pronoun_ratio)
        self.df["readability"] = self.df["clean_text"].apply(textstat.flesch_reading_ease)
        self.df["word_count"] = self.df["clean_text"].apply(lambda x: len(x.split()))
        return self

    def extract_tfidf(self):
        vec = TfidfVectorizer(
            max_features=self.config["tfidf_max_features"],
            ngram_range=(1, 3),
            stop_words="english",
            sublinear_tf=True
        )

        tfidf = vec.fit_transform(self.df["clean_text"])
        tfidf_cols = [f"tfidf_{v}" for v in vec.get_feature_names_out()]

        self.df = pd.concat(
            [self.df, pd.DataFrame(tfidf.toarray(), columns=tfidf_cols)],
            axis=1
        )

        self.tfidf_vectorizer = vec
        return self

    def extract_sbert(self):
        model = SentenceTransformer(self.config["sbert_model"], device="cuda" if torch.cuda.is_available() else "cpu")
        emb = model.encode(self.df["clean_text"].tolist(), batch_size=32, show_progress_bar=True)

        emb_df = pd.DataFrame(emb, columns=[f"sbert_{i}" for i in range(emb.shape[1])])
        self.df = pd.concat([self.df, emb_df], axis=1)
        return self

    def save(self):
        os.makedirs(self.config["save_dir"], exist_ok=True)
        self.df.to_pickle(f"{self.config['save_dir']}/full_features.pkl")
        joblib.dump(self.tfidf_vectorizer, f"{self.config['save_dir']}/tfidf_vectorizer.joblib")
        print("Phase 0 Done! Feature Store Created.")

In [ ]:
pipeline = DepressionFeatureStore(CONF)
pipeline.load_and_clean().extract_psycholinguistic().extract_tfidf().extract_sbert().save()
df = pd.read_pickle("processed_data/full_features.pkl")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.head(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)


df = pd.read_pickle("processed_data/full_features.pkl")

drop_cols = [c for c in df.columns if c.startswith("sbert_")] + ["clean_text"]
X = df.drop(columns=drop_cols)
y = df["is_depression"]


scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

cm_rf = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Depressed", "Depressed"], yticklabels=["Non-Depressed", "Depressed"])
plt.title("Random Forest")
plt.show()

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
print("AUC =", auc(fpr, tpr))

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

path = '/content/Integrating AI-DataSet_Youth_Wellness.xlsx'
df = pd.read_excel(path, sheet_name="Sheet3")

possible_text = ["text", "clean_text", "body", "content", "sentence", "post"]
possible_label = ["label", "target", "class", "sentiment"]

text_col = next((c for c in df.columns if c.lower() in possible_text), df.columns[0])
label_col = next((c for c in df.columns if c.lower() in possible_label), df.columns[-1])

print("\nUsing text column:", text_col)
print("Using label column:", label_col)

# Extract data
X = df[text_col].astype(str)
y = df[label_col]

if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)


rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)


rf_train_prob = rf_model.predict_proba(X_train)
rf_test_prob = rf_model.predict_proba(X_test)

print("\nRF prob shape:", rf_train_prob.shape)

In [ ]:
base_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

ada_model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=200,
    learning_rate=1.0,
    algorithm="SAMME",
    random_state=42
)

ada_model.fit(rf_train_prob, y_train)

y_pred = ada_model.predict(rf_test_prob)

In [ ]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

cm_ada = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(3.5,3))
sns.heatmap(cm_ada, annot=True, fmt="d", cmap="Greens", xticklabels=["Non-Depressed", "Depressed"], yticklabels=["Non-Depressed", "Depressed"])
plt.title("AdaBoost")
plt.show()

In [ ]:
!pip install xgboost lightgbm

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import optuna
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json
import joblib
import torch


SEED = 42

class XGBOnlyPipeline:
    def __init__(self, data_path):
        self.df = pd.read_pickle(data_path)
        self.cm_xgb = None
        self.best_model = None


    def prepare_data(self):
        y = self.df['is_depression']
        drop_cols = ['clean_text', 'is_depression'] + [c for c in self.df if c.startswith("sbert_")]
        X = self.df.drop(columns=drop_cols)

        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=SEED
        )

    def objective(self, trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3),
            "tree_method": "hist",
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "random_state": SEED
        }

        model = xgb.XGBClassifier(**params)
        model.fit(self.X_train, self.y_train)
        preds = model.predict(self.X_test)
        return f1_score(self.y_test, preds, average="macro")

    def run(self):
        print("Starting Optuna...")
        study = optuna.create_study(direction="maximize")
        study.optimize(self.objective, n_trials=30)

        print("Best Params:", study.best_params)

        self.best_model = xgb.XGBClassifier(
            **study.best_params,
            tree_method="hist",
            device="cuda" if torch.cuda.is_available() else "cpu",
            random_state=SEED
        )

        self.best_model.fit(self.X_train, self.y_train)

        preds = self.best_model.predict(self.X_test)

        print(classification_report(self.y_test, preds))


        self.cm_xgb = confusion_matrix(self.y_test, preds)


        os.makedirs("xgb_results", exist_ok=True)
        joblib.dump(self.best_model, "xgb_results/model.joblib")

        with open("xgb_results/best_params.json", "w") as f:
            json.dump(study.best_params, f, indent=4)


        np.save("xgb_results/confusion_matrix.npy", self.cm_xgb)

        print("XGBoost training complete.")

    def plot_confusion_matrix(self):
        if self.cm_xgb is None:
            print("Confusion matrix not found. Run model first.")
            return

        plt.figure(figsize=(6, 4))
        sns.heatmap(
            self.cm_xgb,
            annot=True,
            fmt="d",
            cmap="Greens",
            xticklabels=["Non-Depressed", "Depressed"],
            yticklabels=["Non-Depressed", "Depressed"]
        )
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("XGBoost")
        plt.show()



xgb_pipeline = XGBOnlyPipeline("processed_data/full_features.pkl")
xgb_pipeline.prepare_data()
xgb_pipeline.run()
xgb_pipeline.plot_confusion_matrix()

In [ ]:
!pip install lightgbm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix


from lightgbm import LGBMClassifier

path = '/content/Integrating AI-DataSet_Youth_Wellness.xlsx'
df = pd.read_excel(path, sheet_name="Sheet3")


df_cleaned = df.copy()

print("Columns available:", df_cleaned.columns)
df_cleaned.head()

X_text_clean = df_cleaned['clean_text'].astype(str).values
y_clean = df_cleaned['is_depression'].values

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_text_clean,
    y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

vectorizer_clean = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_vec_clean = vectorizer_clean.fit_transform(X_train_clean)
X_test_vec_clean = vectorizer_clean.transform(X_test_clean)

model_lgbm = LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42
)

model_lgbm.fit(X_train_vec_clean, y_train_clean)

y_pred_lgbm = model_lgbm.predict(X_test_vec_clean)

from sklearn.metrics import classification_report

print("Classification Report (LightGBM):")
print(classification_report(y_test_clean, y_pred_lgbm))

cm_lgbm = confusion_matrix(y_test_clean, y_pred_lgbm)

plt.figure(figsize=(3.5, 3))
sns.heatmap(
    cm_lgbm,
    annot=True,
    fmt='d',
    cmap='Greens',
    linewidths=1,
    linecolor='white',
    xticklabels=["Non-Depressed", "Depressed"],
    yticklabels=["Non-Depressed", "Depressed"]
)

plt.title("LightGBM)", fontsize=16, weight="bold")
plt.xlabel("Predicted", fontsize=14)
plt.ylabel("True", fontsize=14)
plt.show()

print("Accuracy (LightGBM):", accuracy_score(y_test_clean, y_pred_lgbm))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_all_confusion_matrices(cm_dict):
    n = len(cm_dict)
    cols = n
    rows = 1

    plt.figure(figsize=(cols * 3.2, rows * 3.2))

    for idx, (name, cm) in enumerate(cm_dict.items(), 1):
        plt.subplot(rows, cols, idx)
        ax = sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Greens",
            cbar=False,
            annot_kws={"size": 14},
            xticklabels=["ND", "DP"],
            yticklabels=["ND", "DP"],
            linewidths=1,
            linecolor="black",
            square=True
        )

        plt.title(name, fontsize=14, fontweight="bold")
        plt.xlabel("Predicted Label", fontsize=12, fontweight="bold")
        plt.ylabel("Actual Label", fontsize=12, fontweight="bold")
        plt.xticks(fontsize=11, fontweight="bold")
        plt.yticks(fontsize=11, rotation=0, fontweight="bold")

        for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_linewidth(1.5)
                spine.set_color("black")

    plt.tight_layout()
    plt.savefig("all_confusion_matrices.png") # Save the figure
    plt.show()

confusion_matrices = {
    "DT": cm_gini,
    "SVM": cm_svm,
    "RF": cm_rf,
    "AdaBoost": cm_ada,
    "XGBoost": xgb_pipeline.cm_xgb,
    "LightGBM": cm_lgbm
}

plot_all_confusion_matrices(confusion_matrices)